In [ ]:
import sys
from pathlib import Path
from typing import List, Optional, Tuple, Union

project_root = Path.cwd().resolve()
if not (project_root / "scripts").exists():
    project_root = project_root.parent.resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Path to the external DSLPT repository.
# Update this if your local DSLPT checkout lives somewhere else.
repo_path = Path("/home/jocareher/Downloads/DSLPT")
if str(repo_path) not in sys.path:
    sys.path.append(str(repo_path))

import cv2
import numpy as np
import torch
from tqdm.auto import tqdm

from scripts.config import build_config
from scripts.engine.postprocessing import (
    apply_homogeneous_transform,
    project_landmarks_between_sizes,
)
from scripts.inference import DetectorExportInferenceDataset
from model import Dynamic_sparse_alignment_network

from Config.default import _C as cfg


In [ ]:
DSLPT_INPUT_SIZE = (256, 256)


def preprocess_image(image: np.ndarray) -> torch.Tensor:
    """Resize and normalize one aligned crop for DSLPT inference.

    This helper is intentionally notebook-local because it reflects how DSLPT
    expects its input image to be prepared.

    Args:
        image: Crop image loaded with OpenCV in BGR format.

    Returns:
        Float tensor with shape ``(1, 3, 256, 256)`` and values in ``[0, 1]``.
    """
    resized_image = cv2.resize(image, DSLPT_INPUT_SIZE)
    normalized_image = resized_image.astype(np.float32) / 255.0
    tensor_image = torch.from_numpy(normalized_image).permute(2, 0, 1).unsqueeze(0)
    return tensor_image.to(dtype=torch.float32)


def filter_68_landmarks(landmarks: np.ndarray) -> np.ndarray:
    """Extract the DSLPT 68-point subset from the model's 98-point output.

    DSLPT predicts 98 landmarks. For this notebook we keep the same 68-point
    subset used elsewhere in your evaluation pipeline.

    Args:
        landmarks: Raw DSLPT prediction array.

    Returns:
        Array with shape ``(68, 2)`` in DSLPT's normalized coordinate space.
    """
    mapping_indices = [
        0, 2, 3, 4, 7, 9, 12, 14, 16, 18, 20, 22, 25, 27, 29, 31, 32,
        33, 34, 35, 36, 37, 42, 43, 44, 45, 46, 51, 52, 53, 54, 55,
        56, 57, 58, 59, 60, 61, 63, 64, 65, 67, 68, 69, 71, 72, 73,
        75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89,
        90, 91, 92, 93, 94, 95,
    ]
    final_landmarks = landmarks[-1]
    return final_landmarks[mapping_indices, :]


def normalized_to_pixel_landmarks(
    landmarks: np.ndarray,
    image_size: Tuple[int, int],
) -> np.ndarray:
    """Convert normalized DSLPT landmarks into absolute pixel coordinates.

    Args:
        landmarks: Landmark coordinates normalized to ``[0, 1]``.
        image_size: ``(width, height)`` of the image space those normalized
            landmarks should be expressed in.

    Returns:
        Array with shape ``(68, 2)`` in absolute pixel coordinates.
    """
    image_width, image_height = image_size
    landmarks_px = landmarks.astype(np.float32).copy()
    landmarks_px[:, 0] *= float(image_width)
    landmarks_px[:, 1] *= float(image_height)
    return landmarks_px


def get_landmark_groups_68() -> List[List[int]]:
    """Return the standard 68-landmark facial connectivity."""
    return [
        list(range(0, 17)),
        list(range(17, 22)),
        list(range(22, 27)),
        list(range(27, 31)),
        list(range(31, 36)),
        [36, 37, 38, 39, 40, 41, 36],
        [42, 43, 44, 45, 46, 47, 42],
        [48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 48],
        [60, 61, 62, 63, 64, 65, 66, 67, 60],
    ]


def draw_landmark_connections(
    image: np.ndarray,
    landmarks_px: np.ndarray,
    color: Tuple[int, int, int],
    line_thickness: int,
) -> None:
    """Draw 68-landmark connection lines in absolute pixel coordinates."""
    for group in get_landmark_groups_68():
        for start_idx, end_idx in zip(group[:-1], group[1:]):
            x1, y1 = landmarks_px[start_idx]
            x2, y2 = landmarks_px[end_idx]
            cv2.line(
                image,
                (int(round(x1)), int(round(y1))),
                (int(round(x2)), int(round(y2))),
                color,
                line_thickness,
            )


def draw_landmarks(
    image: np.ndarray,
    landmarks_px: np.ndarray,
    color: Tuple[int, int, int] = (255, 255, 0),
) -> np.ndarray:
    """Draw 68 landmarks and their connections on one image.

    Args:
        image: Target image in OpenCV BGR format.
        landmarks_px: Landmark coordinates already expressed in the same pixel
            space as ``image``.
        color: BGR color used for both points and connection lines.

    Returns:
        Copy-ready image array with the overlay drawn on top.
    """
    height, width, _ = image.shape
    diagonal = float((width ** 2 + height ** 2) ** 0.5)

    radius = max(4, min(int(diagonal * 0.01), 12))
    line_thickness = max(1, int(radius * 0.6))

    draw_landmark_connections(
        image=image,
        landmarks_px=landmarks_px,
        color=color,
        line_thickness=line_thickness,
    )

    for x_coord, y_coord in landmarks_px:
        cv2.circle(
            image,
            (int(round(x_coord)), int(round(y_coord))),
            radius,
            color,
            -1,
        )

    return image


def save_landmarks_to_txt(landmarks_px: np.ndarray, output_txt_path: Path) -> None:
    """Save one ``x y`` pair per line in absolute image coordinates.

    This matches the plain-text format you asked for:

        x1 y1
        x2 y2
        ...

    Args:
        landmarks_px: Landmark coordinates in the final image space you want to
            evaluate against. In this notebook that means original-image space.
        output_txt_path: Destination ``.txt`` file.
    """
    output_txt_path.parent.mkdir(parents=True, exist_ok=True)
    with output_txt_path.open("w", encoding="utf-8") as file:
        for x_coord, y_coord in landmarks_px.astype(np.float32):
            file.write(f"{x_coord:.6f} {y_coord:.6f}\n")


def run_dslpt_inference(
    image: np.ndarray,
    model: torch.nn.Module,
    device: torch.device,
) -> np.ndarray:
    """Run DSLPT on one aligned crop.

    Args:
        image: Detector-exported aligned face crop in OpenCV BGR format.
        model: Loaded DSLPT model.
        device: Torch device used for inference.

    Returns:
        Array with shape ``(68, 2)`` in absolute pixel coordinates in DSLPT's
        network-input space, which in this notebook is always ``256x256``.
    """
    input_tensor = preprocess_image(image).to(device)

    with torch.inference_mode():
        output_list, _, _, _ = model(input_tensor)
        landmarks_98 = output_list[-1].squeeze().cpu().numpy()

    landmarks_68_normalized = filter_68_landmarks(landmarks_98)
    return normalized_to_pixel_landmarks(
        landmarks=landmarks_68_normalized,
        image_size=DSLPT_INPUT_SIZE,
    )


def process_detector_export_directory(
    export_root: Union[str, Path],
    output_dir: Union[str, Path],
    model: torch.nn.Module,
    device: torch.device,
    source_root: Optional[Union[str, Path]] = None,
    save_crop_overlays: bool = False,
    landmark_color: Tuple[int, int, int] = (255, 255, 0),
) -> dict:
    """Run DSLPT on detector-export crops and save predictions in original space.

    Expected detector-export layout under ``export_root``::

        export_root/
            images/
                crop_0001.png
                crop_0002.png
                ...
            metadata/
                crop_0001.json
                crop_0002.json
                ...

    Each metadata JSON is expected to include at least:
    ``source_image_path`` and ``transform_crop_to_orig``.

    The geometry flow is:

    1. DSLPT predicts landmarks in its own 256x256 input space.
    2. ``project_landmarks_between_sizes`` rescales them to the exported crop size.
    3. ``apply_homogeneous_transform`` maps them from crop space to the original
       image space using the stored ``transform_crop_to_orig`` matrix.

    Args:
        export_root: Root directory of the detector export. This should be the
            folder that contains ``images/`` and ``metadata/``.
        output_dir: Directory where the notebook will create
            ``predictions/labels`` and ``predictions/images``.
        model: Loaded DSLPT model.
        device: Torch device used for inference.
        source_root: Optional base directory used only when the metadata stores
            ``source_image_path`` as a relative path that cannot be resolved from
            ``export_root`` alone.
            Leave this as ``None`` when the metadata already contains absolute
            paths, or when the relative paths are already valid from the export.
            Example: if metadata contains ``source_image_path: \"images_raw/a.jpg\"``
            and the real file lives at ``/data/project/images_raw/a.jpg``, set
            ``source_root=Path(\"/data/project\")``.
        save_crop_overlays: If ``True``, also save qualitative overlays in crop
            coordinates under ``predictions/crops``.
        landmark_color: BGR overlay color for points and connections.

    Returns:
        Summary dictionary with output paths and processing counts.
    """
    config = build_config()
    config.image_size = (DSLPT_INPUT_SIZE[1], DSLPT_INPUT_SIZE[0])
    config.normalization_mean = (0.0, 0.0, 0.0)
    config.normalization_std = (1.0, 1.0, 1.0)

    dataset = DetectorExportInferenceDataset(
        export_root=export_root,
        config=config,
        source_root=source_root,
    )

    output_dir = Path(output_dir)
    predictions_dir = output_dir / "predictions"
    prediction_labels_dir = predictions_dir / "labels"
    prediction_overlays_dir = predictions_dir / "images"
    prediction_crops_dir = predictions_dir / "crops"

    prediction_labels_dir.mkdir(parents=True, exist_ok=True)
    prediction_overlays_dir.mkdir(parents=True, exist_ok=True)
    if save_crop_overlays:
        prediction_crops_dir.mkdir(parents=True, exist_ok=True)

    processed_count = 0
    failed_samples: List[str] = []

    for sample_index in tqdm(range(len(dataset)), desc="Processing"):
        try:
            sample = dataset[sample_index]
            metadata = sample["metadata"]

            sample_id = str(metadata["sample_id"])
            crop_image_path = Path(metadata["crop_image_path"])
            source_image_path = Path(metadata["source_image_path"])
            crop_size = tuple(int(value) for value in metadata["crop_size"])
            transform_crop_to_orig = metadata["transform_crop_to_orig"]

            crop_image = cv2.imread(str(crop_image_path))
            if crop_image is None:
                raise FileNotFoundError(f"Could not load crop image: {crop_image_path}")

            predicted_landmarks_network = run_dslpt_inference(
                image=crop_image,
                model=model,
                device=device,
            )
            predicted_landmarks_crop = project_landmarks_between_sizes(
                landmarks=torch.from_numpy(predicted_landmarks_network),
                source_size=(DSLPT_INPUT_SIZE[1], DSLPT_INPUT_SIZE[0]),
                target_size=crop_size,
            ).cpu().numpy()
            predicted_landmarks_original = apply_homogeneous_transform(
                landmarks=predicted_landmarks_crop,
                transform_matrix=transform_crop_to_orig,
            )

            save_landmarks_to_txt(
                landmarks_px=predicted_landmarks_original,
                output_txt_path=prediction_labels_dir / f"{sample_id}.txt",
            )

            original_image = cv2.imread(str(source_image_path))
            if original_image is None:
                raise FileNotFoundError(f"Could not load source image: {source_image_path}")

            overlay_image = draw_landmarks(
                image=original_image.copy(),
                landmarks_px=predicted_landmarks_original,
                color=landmark_color,
            )
            overlay_suffix = source_image_path.suffix or ".png"
            cv2.imwrite(
                str(prediction_overlays_dir / f"{sample_id}{overlay_suffix}"),
                overlay_image,
            )

            if save_crop_overlays:
                crop_overlay = draw_landmarks(
                    image=crop_image.copy(),
                    landmarks_px=predicted_landmarks_crop,
                    color=landmark_color,
                )
                crop_suffix = crop_image_path.suffix or ".png"
                cv2.imwrite(
                    str(prediction_crops_dir / f"{sample_id}{crop_suffix}"),
                    crop_overlay,
                )

            processed_count += 1
        except Exception as error:
            sample_name = f"index_{sample_index}"
            failed_samples.append(sample_name)
            print(f"Failed to process {sample_name}: {error}")

    print("\n=== Processing Summary ===")
    print(f"Total samples processed: {processed_count}")
    print(f"Total samples failed: {len(failed_samples)}")
    if failed_samples:
        print("Failed samples:")
        for sample_name in failed_samples:
            print(f"  {sample_name}")

    return {
        "num_samples": processed_count,
        "predictions_dir": predictions_dir,
        "prediction_labels_dir": prediction_labels_dir,
        "prediction_overlays_dir": prediction_overlays_dir,
        "prediction_crop_overlays_dir": prediction_crops_dir if save_crop_overlays else None,
        "failed_samples": failed_samples,
    }


In [ ]:
# export_root should point to the detector-export root, not directly to the crop images folder.
# It must contain:
#   export_root/images/
#   export_root/metadata/
export_root = Path('/home/jocareher/Documents/baby_face_72')

# source_root is only needed when metadata uses relative source_image_path values
# that cannot be resolved from export_root.
# Most of the time you can keep this as None.
# Example:
# source_root = Path('/home/jocareher/Documents')
source_root = None

# Outputs will be written under:
#   output_directory/predictions/labels
#   output_directory/predictions/images
output_directory = Path('/home/jocareher/Documents/results_dslpt')

# Load DSLPT model.
model_path = Path('/home/jocareher/Downloads/DSLPT_WFLW_6_layers.pth')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Dynamic_sparse_alignment_network(
    num_point=98,
    d_model=256,
    trainable=False,
    return_interm_layers=False,
    nhead=8,
    feedforward_dim=1024,
    initial_path='/home/jocareher/Downloads/DSLPT/Config/init_98.npz',
    cfg=cfg,
)
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

# Run inference on detector-exported aligned crops.
# Final .txt files are saved in original image coordinates.
summary = process_detector_export_directory(
    export_root=export_root,
    output_dir=output_directory,
    model=model,
    device=device,
    source_root=source_root,
    save_crop_overlays=False,
)

summary
